In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
dataType = "UpdraftAreaAverages"
# dataType = "UpdraftAreaAverages_Interpolation" #*TESTING

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
# Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
# Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"#;spinup_hours="-16"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24" 

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

# RunType = (Region,Case,"TEMPO",spinup_hours)
# ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetLoopElements(start_job,end_job):
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job].tolist()
    return loop_elements
loop_elements = GetLoopElements(start_job,end_job)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def InitiateMatrix(variableSubset, fill_nan=False):
    """
    Initializes an output matrix for a given variable subset.
    """
    if "nVertLevels" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzc)

    elif "nVertLevelsP1" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzf)

    else:  # 2D variable case
        shape = (ModelData.Ntime, 1)

    fill_value = np.nan if fill_nan else 0
    output = np.full(shape, fill_value, dtype=float)

    return output

def GetMean(variableSubset):
    #(1/A) times integral of phi dA 
    #dA is [(R*cos(Lat)dLon)][RdLat] = R^2 cos(Lat)dLatdLon ==> weight is simply cos(Lat)
    weights = np.cos(np.deg2rad(variableSubset.latitude))
    variableMean = variableSubset.weighted(weights).mean(
        dim=("latitude", "longitude"),
        skipna=True
    )
    return variableMean

def MeanDBZ(variableSubset):
    # Convert from dBZ → linear Z (mm^6 m^-3)
    variableSubset_power = 10 ** (variableSubset / 10.0)

    # Take mean in linear space
    variableMean = GetMean(variableSubset_power)

    # Convert mean Z → back to dBZ
    variableMean = 10.0 * np.log10(variableMean)

    return variableMean

In [ ]:
#Loading Radar Mask
RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def CalculateCondensateThreshold(ModelData,t,
                                 qt_min=0):
    data = ModelData.GetDataTimestep(t)
    qt = data['qc'] + data['qi'] + data['qg'] + data['qr']
    qtMask = qt > qt_min
    return qtMask

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def InterpolateWToCenters(wData):
    wData_center = 0.5 * (
        wData.isel(nVertLevelsP1=slice(0, -1)) +
        wData.isel(nVertLevelsP1=slice(1, None))
    )
    wData_center = wData_center.rename({"nVertLevelsP1": "nVertLevels"})
    return wData_center

In [ ]:
def RunCalculations(ModelData, varNames, loop_elements,zTarget=None):
    wthresh_updraft = 0.1; wthresh_downdraft = -0.1 #*NEW
    
    outputDictionary={}
    for count, t in enumerate(tqdm(loop_elements, desc="Processing timesteps")):
        # if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
            
        #Loading Data
        [dataSubset,dataSubset_diag,dataSubset_static, lat,lon,zGrid_f,zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

        #Loading W for Later Subsetting
        wSubset_f= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, "w") #*NEW
        wSubset_c = InterpolateWToCenters(wSubset_f) #*will need to fix this in the future
        # #Interpolating Z levels #*TESTING
        # #################################
        # if any(dim.startswith("nVertLevels") for dim in wSubset_f.dims):
        #     if zTarget is None:
        #         [zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c)
        #         zTarget = "loaded"
        #     wSubset_f = ModelData.InterpolateVertical(wSubset_f,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
        #     wSubset_c = ModelData.InterpolateVertical(wSubset_c,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
        # #################################
        wSubset_c = wSubset_c.where(RadarDataMask == True) #*NEW
        wUpdraftThreshold = wSubset_c > wthresh_updraft
        wDowndraftThreshold  = wSubset_c < wthresh_downdraft

        qtMask = CalculateCondensateThreshold(ModelData, t) #*NEW
    
        for varName in varNames:
            if count == 0: print(f"Running for {varName}")
                
            #Subsetting Data
            if varName in ["w"]:
                variableSubset = wSubset_c.copy(deep=False)
            else:
                variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
            
            # #Interpolating Z levels #*TESTING
            # #################################
            # if any(dim.startswith("nVertLevels") for dim in variableSubset.dims):
            #     variableSubset = ModelData.InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
            # #################################

            if varName in ['refl10cm','refl10cm_1km']:
                variableSubset = variableSubset.where(variableSubset > 0)

            #Applying RadarDataMask
            variableSubset = variableSubset.where(RadarDataMask == True)

            #Applying Updraft/Downdraft Thresholds #*NEW
            variableSubset_Updraft = variableSubset.where(wUpdraftThreshold==True).where(qtMask==True) #*NEW
            variableSubset_Downdraft = variableSubset.where(wDowndraftThreshold==True).where(qtMask==True) #*NEW
            
            #Initializing Output
            if count == 0:
                output_Updraft = InitiateMatrix(variableSubset, fill_nan=False) #*NEW
                outputDictionary[f"{varName}_updraft"] = output_Updraft
                output_Downdraft = InitiateMatrix(variableSubset, fill_nan=False) #*NEW
                outputDictionary[f"{varName}_downdraft"] = output_Downdraft                

            #Taking Mean
            if varName in ['refl10cm','refl10cm_1km']:
                variableMean_Updraft = MeanDBZ(variableSubset_Updraft) #*NEW
                variableMean_Downdraft = MeanDBZ(variableSubset_Downdraft) #*NEW
            else:
                variableMean_Updraft = GetMean(variableSubset_Updraft) #*NEW
                variableMean_Downdraft = GetMean(variableSubset_Downdraft) #*NEW
                
            outputDictionary[f"{varName}_updraft"][t] = variableMean_Updraft #*NEW
            outputDictionary[f"{varName}_downdraft"][t] = variableMean_Downdraft #*NEW
    return outputDictionary

def GetData_Subset(ModelData,t):  
    data = ModelData.GetDataTimestep(t,printout=False)
    data_diag = ModelData.GetDataTimestep_diag(t,printout=False)
    
    [latCenter,lonCenter] = DataOperator_Class.LatLonBoundingBox_Center(region=ModelData.region)
    [latBounds, lonBounds] = DataOperator_Class.LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500)
    dataSubset, lat, lon = DataOperator_Class.LatLonBoundingBox_Subset(data,latBounds, lonBounds)
    dataSubset_diag, _, _ = DataOperator_Class.LatLonBoundingBox_Subset(data_diag,latBounds, lonBounds)
    dataSubset_static, _, _ = DataOperator_Class.LatLonBoundingBox_Subset(ModelData.staticData,latBounds, lonBounds)

    # Lon, Lat = np.meshgrid(lon, lat) #not actually needed to plot
    return dataSubset, dataSubset_diag, dataSubset_static, lat, lon, data, data_diag

In [ ]:
def RunAreaAverages(ModelData,varNames,name, loop_elements):
    filePath = DataOperator_Class.GetOutputFilePath(ModelData, DirectoryManager, outputDirectory, fileName = f"outputDictionary_{name}_{loop_elements[0]}-{loop_elements[-1]+1}.h5")
    
    #loading back in 
    try:
        #loading output
        print("\n")
        outputDictionary = DataSaving_Class.LoadDictionaryFromH5(filePath)
        return outputDictionary
    except Exception as e:
        print(f"Error: {e}")
        
        print("Running Calculation")
        outputDictionary = RunCalculations(ModelData, varNames, loop_elements) #takes about 10 minutes
        #saving output
        print("\n")
        DataSaving_Class.SaveDictionaryToH5(outputDictionary, filePath)
        return outputDictionary

In [ ]:
####################################
#CALCULATING FUNCTIONS

In [ ]:
def GetDictionary_2(ModelData, loop_elements):
    #3D Variables (9 vars)
    #microphysics variables
    varNames = ["qv", "qc+qi", "qr", "qg"]
    #convection variables
    varNames += ["w"]
    
    outputDictionary = RunAreaAverages(ModelData,varNames,"2", loop_elements)
    return outputDictionary

In [ ]:
def RunJob(loop_elements):
    
    #getting NSSL dictionaries
    RunType = (Region,Case,"NSSL",spinup_hours)
    ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, printSummary=False)
    
    outputDictionary_3D_NSSL = GetDictionary_2(ModelData, loop_elements)
    
    #getting TEMPO dictionaries
    RunType = (Region,Case,"TEMPO",spinup_hours)
    ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, printSummary=False)
    
    outputDictionary_3D_TEMPO = GetDictionary_2(ModelData, loop_elements)

    return outputDictionary_3D_NSSL,outputDictionary_3D_TEMPO

In [ ]:
####################################
#CALCULATING
running = True #keep true when job_array is running
# running = False

In [ ]:
if running:
    [outputDictionary_3D_NSSL,outputDictionary_3D_TEMPO] = RunJob(loop_elements)

In [ ]:
####################################
#RECOMBINING
recombining = False #keep false when job_array is running
# recombining = True

In [ ]:
def AddDictionaries(dictA, dictB):
    """
    Modifies dictA by adding dictB values into it
    """
    for key in dictA:
        dictA[key] += dictB[key]
        
def Recombine():
    for job_id in tqdm(range(1, num_jobs + 1)):
    
        start_job, end_job = JobArray._get_job_range(job_id)
        loop_elements = GetLoopElements(start_job, end_job)
    
        dict3D_NSSL, dict3D_TEMPO = RunJob(loop_elements)
    
        if job_id == 1:
            dict3D_NSSL_all = dict3D_NSSL
            dict3D_TEMPO_all = dict3D_TEMPO
        else:
            AddDictionaries(dict3D_NSSL_all, dict3D_NSSL)
            AddDictionaries(dict3D_TEMPO_all,dict3D_TEMPO)
    return dict3D_NSSL_all,dict3D_TEMPO_all

In [ ]:
if recombining:
    [outputDictionary_3D_NSSL,outputDictionary_3D_TEMPO] = Recombine()

In [ ]:
####################################
#PLOTTING FUNCTIONS
plotting = False #keep false when job array is running
plotting = True

In [ ]:
def GetVerticalCoord(dataSubset):
    pressure_profile = dataSubset['pressure'].mean(dim=("latitude","longitude")).data
    dp = pressure_profile[-1] - pressure_profile[-2]
    p_topface = pressure_profile[-1] + dp  # extrapolate linearly
    pressure_profile_face = np.append(pressure_profile, p_topface)
    return (pressure_profile/100,pressure_profile_face/100)

if plotting:
    [dataSubset,dataSubset_diag,dataSubset_static, lat,lon,zGrid_f,zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t=0)
    pressure_profiles = GetVerticalCoord(dataSubset)
    time_strings = ModelData.timeStrings
    time = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]

In [ ]:
#Helper Functions

# Example: align datetime x-limits to min/max of your data
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    import numpy as np
    from matplotlib.dates import date2num

    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass

    ax.set_xlim(times.min(), times.max())

In [ ]:
def variableLinePlot_Vertical(axis, output,zlevels_plot, varName, units, color, label):
    axis.plot(output.squeeze(),zlevels_plot, color=color, label=label)
    axis.set_xlabel(f"{varName} " + fr"$({units})$")
    axis.set_ylabel("z (km)")
    axis.grid(True)

def variableLinePlot_Time(axis, time, output, varName, units, color, label):
        axis.plot(time, output.squeeze(), color=color, label=label)
        axis.set_ylabel(f"{varName} " + fr"$({units})$")
        axis.set_xlabel("Time")
        axis.grid(True)
        SetXLimitsDatetime(axis, time)
        axis.tick_params(axis="x", rotation=45)
        axis.legend()

def FixAxisLabels_Contour(axis, varName, 
                          zlevels_plot):
    top = 0.3 if "qr" in varName else maxZLevel
    if ModelData.region=="Hawaii" and ModelData.case=="TRADES":
        top = 1 if "qr" in varName else maxZLevel

    bottom = np.min(zlevels_plot)
    axis.set_ylim(bottom,top)    
    
    axis.set_xlabel("Time")
    axis.set_ylabel("z (km)")
    axis.tick_params(axis="x", rotation=45)

def FixAxisLabels_Line_Vertical(axis, varName,updowndraft,
                       zlevels_plot,
                       useTitle=True):
    top = 0.3 if "qr" in varName else maxZLevel
    if ModelData.region=="Hawaii" and ModelData.case=="TRADES":
        top = 1 if "qr" in varName else maxZLevel
        
    bottom = np.min(zlevels_plot)
    axis.set_ylim(bottom,top)    

    if useTitle: axis.set_title(f"Vertical profiles ({updowndraft})")
    axis.set_ylabel("z (km)")
    axis.legend()

In [ ]:
# PlotUpdraftDowndraftPlots

# zGrid_f, zGrid_c = ModelData.GetZGrids() #*TESTING
# [zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c) #*TESTING

def PlotUpdraftDowndraftPlots(varName='w',units="m/s",
                              multiplier=1,
                              num_levels=19):

    #Z AXIS
    z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
    zlevels = np.loadtxt(z_levels_filePath)/1e3
    zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
    if varName not in ['w']:
        zlevels_plot = zlevels_center.copy()
        # zlevels_plot = zTarget_c/1e3 #*TESTING
    else:
        # zlevels_plot = zlevels.copy()
        zlevels_plot = zlevels_center.copy()
        ## zlevels_plot = zTarget_f/1e3 #*TESTING
        # zlevels_plot = zTarget_c/1e3 #*TESTING
    
    fig, axes = plt.subplots(
        nrows=2, ncols=5,
        figsize=(18, 8),
        constrained_layout=True
    )
    
    # =========================
    # ROW 1 — UPDRAFT
    # =========================
    updowndraft = "updraft"
    
    a = multiplier*outputDictionary_3D_NSSL[f"{varName}_{updowndraft}"]
    b = multiplier*outputDictionary_3D_TEMPO[f"{varName}_{updowndraft}"]
    c = a - b

    #Levels
    vmin = np.nanmin([np.nanmin(a), np.nanmin(b)])
    vmax = np.nanmax([np.nanmax(a), np.nanmax(b)])
    # levels = np.linspace(vmin,vmax,num_levels)
    levels = np.round(np.linspace(vmin,vmax,num_levels),3)
    diff_vlim = np.nanmax(np.abs(c))
    diff_levels = np.linspace(-diff_vlim, diff_vlim, num_levels)
    diff_norm = TwoSlopeNorm(vcenter=0.0, vmin=-diff_vlim, vmax=diff_vlim)


    #Plotting
    axis = axes[0, 0]
    cf1 = axis.contourf(time, zlevels_plot, a.T, levels=levels)
    axis.set_title("NSSL (updraft)")
    plt.colorbar(cf1, ax=axes[0, 0], pad=-0.05)
    FixAxisLabels_Contour(axis, varName,
                          zlevels_plot)

    axis = axes[0, 1] 
    cf2 = axis.contourf(time, zlevels_plot, b.T, levels=levels)
    axis.set_title("TEMPO (updraft)")
    plt.colorbar(cf2, ax=axes[0, 1], pad=-0.05)
    FixAxisLabels_Contour(axis, varName,
                          zlevels_plot)

    axis = axes[0, 2]
    cf3 = axis.contourf(time, zlevels_plot, c.T, levels=diff_levels,
                              cmap="RdBu_r", norm=diff_norm)
    axes[0, 2].set_title("NSSL − TEMPO (updraft)")
    plt.colorbar(cf3, ax=axes[0, 2], pad=-0.05)
    FixAxisLabels_Contour(axis, varName,
                          zlevels_plot)
    
    aMean_Vertical = np.nanmean(a, axis=0)
    bMean_Vertical = np.nanmean(b, axis=0)
    aMean_Time = np.nanmean(a, axis=1)
    bMean_Time = np.nanmean(b, axis=1)
    
    axis = axes[0, 3]
    variableLinePlot_Vertical(axis, aMean_Vertical,zlevels_plot, varName, units=units, color="blue", label="NSSL")
    variableLinePlot_Vertical(axis, bMean_Vertical,zlevels_plot, varName, units=units, color="green", label="TEMPO")
    # axis.axhline(1) #*TESTING 
    FixAxisLabels_Line_Vertical(axis, varName,"updraft",
                       zlevels_plot)

    aMean = np.nanmean(a, axis=1)
    bMean = np.nanmean(b, axis=1)
    
    axis = axes[0, 4]
    variableLinePlot_Time(axis, time, aMean_Time, varName, units=units, color="blue", label="NSSL")
    variableLinePlot_Time(axis, time, bMean_Time, varName, units=units, color="green", label="TEMPO")

    
    # =========================
    # ROW 2 — DOWNDRAFT
    # =========================
    updowndraft = "downdraft"
    
    a = multiplier*outputDictionary_3D_NSSL[f"{varName}_{updowndraft}"]
    b = multiplier*outputDictionary_3D_TEMPO[f"{varName}_{updowndraft}"]
    c = a - b

    #Levels
    vmin = np.nanmin([np.nanmin(a), np.nanmin(b)])
    vmax = np.nanmax([np.nanmax(a), np.nanmax(b)])
    # levels = np.linspace(vmin,vmax,num_levels)
    levels = np.round(np.linspace(vmin,vmax,num_levels),3)
    diff_vlim = np.nanmax(np.abs(c))
    diff_levels = np.linspace(-diff_vlim, diff_vlim, num_levels)
    diff_norm = TwoSlopeNorm(vcenter=0.0, vmin=-diff_vlim, vmax=diff_vlim)

    #Plotting
    axis = axes[1, 0]
    cf1 = axis.contourf(time, zlevels_plot, a.T, levels=levels)
    axis.set_title("NSSL (downdraft)")
    plt.colorbar(cf1, ax=axes[1, 0], pad=-0.05)
    FixAxisLabels_Contour(axis, varName,
                          zlevels_plot)

    axis = axes[1, 1]
    cf2 = axis.contourf(time, zlevels_plot, b.T, levels=levels)
    axis.set_title("TEMPO (downdraft)")
    plt.colorbar(cf2, ax=axes[1, 1], pad=-0.05)
    FixAxisLabels_Contour(axis, varName,
                          zlevels_plot)

    axis = axes[1, 2]
    cf3 = axis.contourf(time, zlevels_plot, c.T, levels=diff_levels,
                              cmap="RdBu_r",norm=diff_norm)
    axis.set_title("NSSL − TEMPO (downdraft)")
    plt.colorbar(cf3, ax=axes[1, 2], pad=-0.05)
    FixAxisLabels_Contour(axis, varName,
                          zlevels_plot)
    
    aMean_Vertical = np.nanmean(a, axis=0)
    bMean_Vertical = np.nanmean(b, axis=0)
    aMean_Time = np.nanmean(a, axis=1)
    bMean_Time = np.nanmean(b, axis=1)
    
    axis = axes[1, 3]
    variableLinePlot_Vertical(axis, aMean_Vertical,zlevels_plot, varName, units=units, color="blue", label="NSSL")
    variableLinePlot_Vertical(axis, bMean_Vertical,zlevels_plot, varName, units=units, color="green", label="TEMPO")
    FixAxisLabels_Line_Vertical(axis, varName,"downdraft",
                       zlevels_plot)
    axis = axes[1, 4]
    variableLinePlot_Time(axis, time, aMean_Time, varName, units=units, color="blue", label="NSSL")
    variableLinePlot_Time(axis, time, bMean_Time, varName, units=units, color="green", label="TEMPO")
    
    return fig

In [ ]:
def SaveFigure(fig, varName):
    """
    Saves a figure to the appropriate directory based on the models.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{ModelData.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFile = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"UpdraftAreaAverages_{varName}.png"
    )

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
#Removing Levels Above maxZLevel km from Timeseries Averages

maxZLevel = 16
def SubsetAltitude(Dictionary,
                   maxZLevel=16):

    for varName, dataArray in Dictionary.items(): 
        
        #Getting Z Threshold Indexes
        z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
        zlevels = np.loadtxt(z_levels_filePath)/1e3
        zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
        zc_level = np.where(zlevels_center>maxZLevel)[0][0]
        # zlevels = zTarget_f #*TESTING
        # zlevels_center = zTarget_c #*TESTING
        
        zf_level = np.where(zlevels>maxZLevel)[0][0]
        z_level = zf_level if varName == "w" else zc_level
        
        #Applying Nan to Altitudes Greater than maxZLevel km
        
        Dictionary[varName][:,z_level+1:] = np.nan

    return Dictionary

if plotting:
    outputDictionary_3D_NSSL = SubsetAltitude(outputDictionary_3D_NSSL, maxZLevel)
    outputDictionary_3D_TEMPO = SubsetAltitude(outputDictionary_3D_TEMPO, maxZLevel)

In [ ]:
####################################
#PLOTTING

In [ ]:
if plotting:
    varName = "w"; units = "m/s"; multiplier = 1
    fig = PlotUpdraftDowndraftPlots(varName,units,multiplier)
    # SaveFigure(fig, varName)

In [ ]:
if plotting:
    varName = 'qc+qi'; units = "g/kg"; multiplier = 1e3
    fig = PlotUpdraftDowndraftPlots(varName,units,multiplier)
    # SaveFigure(fig, varName)

In [ ]:
if plotting:
    varName = 'qr'; units = "g/kg"; multiplier = 1e3
    fig = PlotUpdraftDowndraftPlots(varName,units,multiplier)
    # SaveFigure(fig, varName)

In [ ]:
####################################
#FINAL PLOTTING
plotting = False #keep false when job array is running
# plotting = True

In [ ]:
def PlotUpdraftDowndraft_ProfilesOnly_MultiVar_Vertical(
    varNames,
    unitsList,
    multiplierList
):

    nVars = len(varNames)

    # =========================
    # FIGURE (NO SHARED AXES)
    # =========================
    fig, axes = plt.subplots(
        nrows=2,
        ncols=nVars,
        figsize=(4 * nVars, 6),
        constrained_layout=True
    )

    # Ensure axes is always 2D
    if nVars == 1:
        axes = axes.reshape(2, 1)

    # =========================
    # Z AXIS (unchanged)
    # =========================
    z_levels_filePath = (
        "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/"
        "MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/"
        "zeta_30km_57levels.txt"
    )
    zlevels = np.loadtxt(z_levels_filePath) / 1e3

    # =========================
    # LOOP OVER VARIABLES (COLUMNS)
    # =========================
    for i, (varName, units, multiplier) in enumerate(
        zip(varNames, unitsList, multiplierList)
    ):

        # --- vertical grid ---
        if varName not in ["w"]:
            zlevels_plot = 0.5 * (zlevels[:-1] + zlevels[1:])
        else:
            zlevels_plot = zlevels.copy()

        # =========================
        # UPDRAFT (row 0)
        # =========================
        a = multiplier * outputDictionary_3D_NSSL[f"{varName}_updraft"]
        b = multiplier * outputDictionary_3D_TEMPO[f"{varName}_updraft"]

        a = a.copy(); b = b.copy()
        aMean_Vertical = np.nanmean(a, axis=0)
        bMean_Vertical = np.nanmean(b, axis=0)

        axis = axes[0, i]
        variableLinePlot_Vertical(axis, aMean_Vertical, zlevels_plot,
                         varName, units, color="blue", label="NSSL")
        variableLinePlot_Vertical(axis, bMean_Vertical, zlevels_plot,
                         varName, units, color="green", label="TEMPO")

        FixAxisLabels_Line_Vertical(axis, varName, "updraft", zlevels_plot,
                           useTitle=False)

        # =========================
        # DOWNDRAFT (row 1)
        # =========================
        a = multiplier * outputDictionary_3D_NSSL[f"{varName}_downdraft"]
        b = multiplier * outputDictionary_3D_TEMPO[f"{varName}_downdraft"]

        aMean_Vertical = np.nanmean(a, axis=0)
        bMean_Vertical = np.nanmean(b, axis=0)

        axis = axes[1, i]
        variableLinePlot_Vertical(axis, aMean_Vertical, zlevels_plot,
                         varName, units, color="blue", label="NSSL")
        variableLinePlot_Vertical(axis, bMean_Vertical, zlevels_plot,
                         varName, units, color="green", label="TEMPO")

        FixAxisLabels_Line_Vertical(axis, varName, "downdraft", zlevels_plot,
                           useTitle=False)

    

    return fig

In [ ]:
def PlotUpdraftDowndraft_ProfilesOnly_MultiVar_Time(
    varNames,
    unitsList,
    multiplierList
):

    nVars = len(varNames)

    # =========================
    # FIGURE (NO SHARED AXES)
    # =========================
    fig, axes = plt.subplots(
        nrows=2,
        ncols=nVars,
        figsize=(4 * nVars, 6),
        constrained_layout=True
    )

    # Ensure axes is always 2D
    if nVars == 1:
        axes = axes.reshape(2, 1)

    # =========================
    # Z AXIS (unchanged)
    # =========================
    z_levels_filePath = (
        "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/"
        "MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/"
        "zeta_30km_57levels.txt"
    )
    zlevels = np.loadtxt(z_levels_filePath) / 1e3

    # =========================
    # LOOP OVER VARIABLES (COLUMNS)
    # =========================
    for i, (varName, units, multiplier) in enumerate(
        zip(varNames, unitsList, multiplierList)
    ):

        # --- vertical grid ---
        if varName not in ["w"]:
            zlevels_plot = 0.5 * (zlevels[:-1] + zlevels[1:])
        else:
            zlevels_plot = zlevels.copy()


        # =========================
        # UPDRAFT (row 0)
        # =========================
        a = multiplier * outputDictionary_3D_NSSL[f"{varName}_updraft"]
        b = multiplier * outputDictionary_3D_TEMPO[f"{varName}_updraft"]

        aMean_Time = np.nanmean(a, axis=1)
        bMean_Time = np.nanmean(b, axis=1)

        axis = axes[0, i]
        variableLinePlot_Time(axis, time, aMean_Time, varName, units=units, color="blue", label="NSSL")
        variableLinePlot_Time(axis, time, bMean_Time, varName, units=units, color="green", label="TEMPO")

        # =========================
        # DOWNDRAFT (row 1)
        # =========================
        a = multiplier * outputDictionary_3D_NSSL[f"{varName}_downdraft"]
        b = multiplier * outputDictionary_3D_TEMPO[f"{varName}_downdraft"]

        aMean_Time = np.nanmean(a, axis=1)
        bMean_Time = np.nanmean(b, axis=1)

        axis = axes[1, i]
        variableLinePlot_Time(axis, time, aMean_Time, varName, units=units, color="blue", label="NSSL")
        variableLinePlot_Time(axis, time, bMean_Time, varName, units=units, color="green", label="TEMPO")

    
    return fig

In [ ]:
if plotting:
    varNames = ["w", "qc+qi", "qr"]
    units_List = ["m/s", "g/kg", "g/kg"]
    multipliers = [1.0, 1e3, 1e3]
    
    fig = PlotUpdraftDowndraft_ProfilesOnly_MultiVar_Vertical(
        varNames,
        units_List,
        multipliers
    )
    fig.suptitle(f"{ModelData.region} {ModelData.case}",y=1.05,fontweight='bold',fontsize=15)
    SaveFigure(fig, varName="combined_Vertical")

In [ ]:
if plotting:
    varNames = ["w", "qc+qi", "qr"]
    units_List = ["m/s", "g/kg", "g/kg"]
    multipliers = [1.0, 1e3, 1e3]
    
    fig = PlotUpdraftDowndraft_ProfilesOnly_MultiVar_Time(
        varNames,
        units_List,
        multipliers
    )
    fig.suptitle(f"{ModelData.region} {ModelData.case}",y=1.05,fontweight='bold',fontsize=15)
    SaveFigure(fig, varName="combined_Time")

In [ ]:
####################################
#PLOTTING ALL SIMULATIONS
plotting = False #keep false when job array is running
# plotting = True

In [ ]:
def GetFigureFilePath(region,case,spinup_hours,
                      verticalTime="Vertical",
                      extension="png"):
    # --- Define output subdirectory ---
    inputSubDirectory = f"{region}_{case}_{spinup_hours}hrs"
    load_dir = os.path.join(outputPlottingDirectory, inputSubDirectory)
    # --- File path ---
    fileName = f"UpdraftAreaAverages_combined_{verticalTime}"
    inputFilePath = os.path.join(
        load_dir,
        f"{fileName}.{extension}"
    )
    return inputFilePath

def GetFilePaths(verticalTime):
    caseList = ConsolidateFigures_CLASS.GetCaseList()
    filePaths = []
    for region, case, spinup_hours in caseList:
        filePaths.append(GetFigureFilePath(region,case,spinup_hours,
                                           verticalTime))
    return filePaths

In [ ]:
if plotting:
    sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
    from CLASSES_Plotting import ConsolidateFigures_CLASS

In [ ]:
if plotting:
    verticalTime = "Vertical"
    filePaths = GetFilePaths(verticalTime)
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(5, 4),
                                                     wspace=0.02,hspace=0.02,
                                                     dpi=1200)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig,dpi=1200, saveDirectory=outputPlottingDirectory,fileName=dataType+f"_{verticalTime}")

In [ ]:
if plotting:
    verticalTime = "Time"
    filePaths = GetFilePaths(verticalTime)
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(5, 4),
                                                     wspace=0.02,hspace=0.02,
                                                     dpi=1200)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig,dpi=1200, saveDirectory=outputPlottingDirectory,fileName=dataType+f"_{verticalTime}")